In [ ]:
import cv2
import numpy as np
import pickle
from tqdm import tqdm
import os
import json
import random
# from proto import *
from math import sqrt

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Activation, Flatten, Reshape, LeakyReLU
from tensorflow.keras.layers import Conv2D, MaxPooling2D
from tensorflow.keras.preprocessing.image import ImageDataGenerator, img_to_array
from keras.optimizers import Adam

from sklearn.model_selection import train_test_split

import keras.backend as K

In [ ]:
GRID_H, GRID_W = 12, 12

IMAGE_SIZE = (4000, 6000)
EPOCHS = 4
BATCH_SIZE = 2
WARM_UP_BATCHES = 0

NO_OBJECT_SCALE = 1.
OBJECT_SCALE = 5.
COORD_SCALE = 1.

INIT_LR = 1e-3

In [ ]:
img_path = r'..\..\ImageGenerator\Images'
data_path = r'..\..\ImageGenerator\Image Data'

In [ ]:
data = []
labels = []
def create_training_data(image_path, json_path, img_size):
    global data
    global labels
    
    training_data = []
    training_labels = []
    random.seed(27)
    
    for img in tqdm(os.listdir(image_path)):
        try:
            image = cv2.imread(os.path.join(image_path, img))
            image = cv2.resize(image, (img_size, img_size))
        except: 
            continue
        image = img_to_array(image)
        training_data.append(image)

        image_grid = np.zeros((GRID_H, GRID_W, 5))
        
        json_file = img[0:-4] + '.json'
        json_dict = json.load(open(os.path.join(json_path, json_file)))
        for target in json_dict:
            x = json_dict[target]['x']
            y = json_dict[target]['y']
            grid_x = int(x * GRID_W)
            grid_y = int(y * GRID_H)
            width = json_dict[target]['width']
            height = json_dict[target]['height']
            grid_cell = np.zeros((1, 1, 5))
            
            grid_cell[0, 0] = np.array([1, x, y, width, height])
            image_grid[grid_y, grid_x] = grid_cell
        training_labels.append(image_grid)
    data = np.array(training_data, dtype="float") / 255.0
    labels = np.array(training_labels, dtype="float")

In [ ]:
def custom_loss(y_true, y_pred):
    mask_shape = tf.shape(y_true)[:3]

    cell_x = tf.cast(tf.reshape(tf.tile(tf.range(GRID_W), [GRID_H]), (1, GRID_H, GRID_W, 1, 1)), tf.float32)
    # 1 and 2 are switched here because it's transposed, so width and height values are switched
    cell_y = tf.transpose(cell_x, (0, 2, 1, 3, 4))

    cell_grid = tf.tile(tf.concat([cell_x, cell_y], -1), [BATCH_SIZE, 1, 1, 5, 1])

    conf_mask = tf.zeros(mask_shape)
    # coord_mask = tf.zeros(mask_shape)

    # seen = tf.Variable(0.)
    # total_recall = tf.Variable(0.)

    pred_box_xy = tf.sigmoid(y_pred[..., 1:3])  # + cell_grid
    pred_box_wh = tf.exp(y_pred[..., 3:])  # * np.reshape(ANCHORS, [1, 1, 1, BOX, 2])

    pred_box_conf = tf.sigmoid(y_pred[..., 4])

    true_box_xy = y_true[..., 1:3]
    true_box_wh = y_true[..., 3:]

    true_wh_half = true_box_wh / 2.
    true_mins = true_box_xy - true_wh_half
    true_maxes = true_box_xy + true_wh_half

    pred_wh_half = pred_box_wh / 2.
    pred_mins = pred_box_xy - pred_wh_half
    pred_maxes = pred_box_xy - pred_wh_half

    intersect_mins = tf.maximum(pred_mins, true_mins)
    intersect_maxes = tf.minimum(pred_maxes, true_maxes)
    intersect_wh = tf.maximum(intersect_maxes - intersect_mins, 0.)
    intersect_areas = intersect_wh[..., 0] * intersect_wh[..., 1]

    true_areas = true_box_wh[..., 0] * true_box_wh[..., 1]
    pred_areas = pred_box_wh[..., 0] * pred_box_wh[..., 1]

    union_areas = pred_areas + true_areas - intersect_areas
    iou_scores = tf.truediv(intersect_areas, union_areas)
    best_ious = tf.reduce_max(iou_scores)

    true_box_conf = iou_scores * y_true[..., 0]

    coord_mask = tf.expand_dims(y_true[..., 0], axis=-1) * COORD_SCALE

    conf_mask = conf_mask + tf.cast(best_ious < 0.6, tf.float32) * (1 - y_true[..., 0]) * NO_OBJECT_SCALE

    conf_mask = conf_mask + y_true[..., 0] * OBJECT_SCALE

    # warm up training
    # no_boxes_mask = tf.cast(coord_mask < COORD_SCALE/2., tf.float32)
    # seen.assign_add(1.)
    #
    # true_box_xy, true_box_wh, coord_mask = tf.cond(tf.less(seen, WARM_UP_BATCHES),
    #                         lambda: [true_box_xy + (0.5 + cell_grid) * no_boxes_mask])

    nb_coord_box = tf.reduce_sum(tf.cast(coord_mask > 0.0, tf.float32))
    nb_conf_box = tf.reduce_sum(tf.cast(conf_mask > 0.0, tf.float32))

    loss_xy = tf.reduce_sum(tf.square(true_box_xy-pred_box_xy) * coord_mask) / (nb_coord_box + 1e-6) / 2.
    loss_wh = tf.reduce_sum(tf.square(true_box_wh-pred_box_wh) * coord_mask) / (nb_coord_box + 1e-6) / 2.
    loss_conf = tf.reduce_sum(tf.square(true_box_conf-pred_box_conf) * conf_mask) / (nb_conf_box + 1e-6) / 2.

    loss = loss_xy + loss_wh + loss_conf

    # print(true_box_xy)
    # nb_true_box = tf.reduce_sum(y_true[..., 0])
    # nb_pred_box = tf.reduce_sum(tf.cast(true_box_conf > 0.5, tf.float32)) * tf.cast(pred_box_conf > 0.3, tf.float32)

    # current_recall = nb_pred_box / (nb_true_box + 1e-6)
    # total_recall.assign_add(current_recall)

    # loss = tf.print(loss, [tf.zeros(1)], message='Dummy line \t', summarize=1000)
    # loss = tf.print(loss, [loss_xy], message='Loss XY \t', summarize=1000)
    # loss = tf.print(loss, [loss_wh], message='Loss WH \t', summarize=1000)
    # loss = tf.print(loss, [loss_conf], message='Loss Conf \t', summarize=1000)
    # loss = tf.print(loss, [loss], message='Total Loss \t', summarize=1000)
    # loss = tf.print(loss, [current_recall], message='Current Recall \t', summarize=1000)
    # loss = tf.print(loss, [total_recall/seen], message='Average Recall \t', summarize=1000)

    return loss

In [ ]:
def create_model(width, height, depth):
    model = Sequential()
    input_shape = (height, width, depth)
    chan_dim = -1
    if K.image_data_format == "channels_first":
        chan_dim = 1
        
    model.add(Conv2D(32, (9, 9), input_shape=input_shape, padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(32, (9, 9), padding='same', strides=2))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(64, (7, 7), padding='same', strides=2))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(64, (7, 7), padding='same', strides=2))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(64, (1, 1), padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(128, (7, 7), padding='same', strides=2))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(128, (7, 7), padding='same', strides=2))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(128, (5, 5), padding='same', strides=2))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(256, (5, 5), padding='same', strides=2))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(256, (5, 5), padding='same', strides=2))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(512, (3, 3), padding='same', strides=2))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(5, (1, 1), padding='same'))
    model.add(Activation('relu'))
    model.summary()


    return model

In [ ]:
def model_training_data():
    (xTrain, xTest, yTrain, yTest) = train_test_split(data, labels, test_size=0.2, random_state=27)
    
#     aug = ImageDataGenerator(rotation_range=25, width_shift_range=0.1, height_shift_range=0.1, \
#                             shear_range=0.2, zoom_range=0.2, fill_mode="nearest")
    
    model = create_model(6000, 6000, 3)
    opt = Adam(lr=INIT_LR, decay=INIT_LR / EPOCHS)
    
    model.compile(loss=custom_loss, optimizer=opt, metrics=["accuracy"])
    
    print("TRAINING NETWORK...")
    model.fit(xTrain, yTrain, validation_data=(xTest, yTest), steps_per_epoch=len(xTrain) // BATCH_SIZE, \
             epochs=EPOCHS, verbose=1)
    
    print("SAVING MODEL...")
    model.save('bounding_box.model', save_format="h5")
    f.open(r"../training_data/bounding_box.pickle", "wb")
    save_data = [data, labels]
    f.write(pickle.dumps(save_data))
    f.close()  
    

In [ ]:
def test_model(imagePath):
    image = cv2.imread(imagePath)
    image = cv2.resize(image, (6000, 6000))
    image = image.astype("float") / 255.0
    image = img_to_array(image)
    image = np.expand_dims(image, axis=0)
    
    print("Classifying Image...")
    prediction = model.predict(image)[0]
    print("RESULT: ")
    print(prediction[0, 0])

In [ ]:
create_training_data(img_path, data_path, 6000)

In [ ]:
with tf.device('CPU:0'):
    model_training_data()

In [ ]:
# tf.config.list_physical_devices()

def create_model(width, height, depth):
    model = Sequential()
    input_shape = (height, width, depth)
    chan_dim = -1
    if K.image_data_format == "channels_first":
        chan_dim = 1
        
    model.add(Conv2D(64, (7, 7), input_shape=input_shape, strides=2))
    model.add(LeakyReLU(alpha=0.05))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=2))

    model.add(Conv2D(192, (5, 5), padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=2))

    model.add(Conv2D(128, (5, 5), padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=2))

    model.add(Conv2D(128, (5, 5), padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=2))

    model.add(Conv2D(256, (3, 3), padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(128, (1, 1), padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(256, (3, 3), padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=2))

    model.add(Conv2D(128, (1, 1), padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(256, (3, 3), padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(128, (1, 1), padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(256, (3, 3), padding='same'))
    model.add(LeakyReLU(alpha=0.05))
#     model.add(Conv2D(128, (3, 3), padding='same'))
#     model.add(LeakyReLU(alpha=0.05))
#     model.add(Conv2D(256, (1, 1), padding='same'))
#     model.add(LeakyReLU(alpha=0.05))
#     model.add(Conv2D(512, (3, 3), padding='same'))
#     model.add(LeakyReLU(alpha=0.05))
    model.add(MaxPooling2D(pool_size=(2, 2), strides=2))

#     model.add(Conv2D(128, (1, 1), padding='same'))
#     model.add(LeakyReLU(alpha=0.05))
#     model.add(Conv2D(512, (3, 3)))
#     model.add(LeakyReLU(alpha=0.05))
#     model.add(Conv2D(512, (3, 3)))
#     model.add(LeakyReLU(alpha=0.05))
#     model.add(Conv2D(512, (3, 3)))

    model.add(Conv2D(512, (3, 3), padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(512, (3, 3), padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(128, (3, 3), strides=2, padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(128, (3, 3), strides=2, padding='same'))
    model.add(LeakyReLU(alpha=0.05))
    model.add(Conv2D(64, (3, 3), strides=2, padding='same'))
    model.add(LeakyReLU(alpha=0.05))

    # model.add(Dense(4096))
    # model.add(Activation('relu'))

    model.add(Dense(GRID_H * GRID_W * 5))
    model.add(Activation('relu'))

#     model.add(Reshape((GRID_H, GRID_W, 5)))
    model.summary()

#     model.compile(loss=custom_loss,
#                   optimizer='adam',
#                   metrics=['accuracy'])
#     model.fit(X, y,
#               batch_size=BATCH_SIZE,
#               epochs=4,
#               validation_split=0.2,
#               )
#     model.save('bounding_box.model')

    return model